# CV2 Final Project - Script Runner (Colab GPU)

This notebook only executes the repository scripts in the same flow as the README:
1. Prepare data
2. Train
3. Evaluate
4. Inference demo

Training with GPU, if you want to use the scripts with cpu or locally check the README for instructions

## 0) Runtime Setup
In Colab, enable GPU first: Runtime > Change runtime type > T4 GPU (or similar).

In [1]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

# Auto-locate the repo in common Colab/Drive locations.
candidates = [
    Path('/content/CV2_Final_Project-1'),
    Path('/content/drive/MyDrive/CV2_Final_Project-1'),
    Path('/content/drive/MyDrive/Colab Notebooks/CV2_Final_Project-1'),
]

PROJECT_ROOT = next(
    (p for p in candidates if (p / 'requirements.txt').exists()),
    None,
 )

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not find project folder. Set PROJECT_ROOT manually to the folder containing requirements.txt'
    )

os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/CV2_Final_Project-1


In [2]:
import torch

if torch.cuda.is_available():
    device_name = "cuda"
elif torch.backends.mps.is_available():
    device_name = "mps"
else:
    device_name = "cpu"
    
device = torch.device(device_name)
print(f"Code runs in {device}")

Code runs in cuda


In [3]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
# Install project dependencies from absolute path
!pip install -r "$PROJECT_ROOT/requirements.txt"

## 1) Prepare Data
Place `maps.zip` in the project root (or adjust the path below).

In [5]:
# CAREFUL, just nneed to run it once
# !python scripts/prepare_data.py --zip maps.zip --out data/raw

## 2) Train

In [6]:
!python scripts/train.py --config configs/base.yaml

Starting training on cuda
Train samples: 350 | Val samples: 75
Epoch [1/100] Step [44/44] D: 0.2615 | G: 12.9748 (GAN: 1.5263, L1: 0.1145)
Epoch [1/100] done | Train D: 0.4503 | Train G: 18.4906 | Val L1: 0.1098
New best checkpoint at epoch 1: checkpoints/best.pt (Val L1: 0.1098)
Epoch [2/100] Step [44/44] D: 0.1981 | G: 15.8683 (GAN: 1.6374, L1: 0.1423)
Epoch [2/100] done | Train D: 0.2117 | Train G: 13.1804 | Val L1: 0.1060
New best checkpoint at epoch 2: checkpoints/best.pt (Val L1: 0.1060)
Epoch [3/100] Step [44/44] D: 0.1907 | G: 14.4268 (GAN: 2.0181, L1: 0.1241)
Epoch [3/100] done | Train D: 0.1292 | Train G: 13.3995 | Val L1: 0.1022
New best checkpoint at epoch 3: checkpoints/best.pt (Val L1: 0.1022)
Epoch [4/100] Step [44/44] D: 0.0661 | G: 14.2835 (GAN: 2.9944, L1: 0.1129)
Epoch [4/100] done | Train D: 0.0988 | Train G: 13.7054 | Val L1: 0.1014
New best checkpoint at epoch 4: checkpoints/best.pt (Val L1: 0.1014)
Epoch [5/100] Step [44/44] D: 0.0570 | G: 14.5780 (GAN: 3.4058, L

## 3) Evaluate

In [7]:
!python scripts/evaluate.py --config configs/base.yaml --checkpoint checkpoints/best.pt

Evaluation complete on 75 test samples.
MAE: 0.102869 | PSNR: 17.7950 | SSIM: 0.3460
Metrics file: outputs/metrics.json


## 4) Inference Demo (Folder)

In [10]:
!python scripts/infer.py --checkpoint checkpoints/best.pt --input data/raw/maps/test --output outputs/demo_test --paired-input

Traceback (most recent call last):
  File "/content/drive/MyDrive/CV2_Final_Project-1/scripts/infer.py", line 157, in <module>
    main()
  File "/content/drive/MyDrive/CV2_Final_Project-1/scripts/infer.py", line 132, in main
    image_paths = list_images(args.input)
                  ^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/CV2_Final_Project-1/scripts/infer.py", line 27, in list_images
    raise FileNotFoundError(f"Input path does not exist: {path}")
FileNotFoundError: Input path does not exist: data/raw/maps/test


## Optional: Inference Demo (Single Image)

In [9]:
!python scripts/infer.py --checkpoint checkpoints/best.pt --input data/raw/maps/test/1.jpg --output outputs/demo_single.png --paired-input

Traceback (most recent call last):
  File "/content/drive/MyDrive/CV2_Final_Project-1/scripts/infer.py", line 157, in <module>
    main()
  File "/content/drive/MyDrive/CV2_Final_Project-1/scripts/infer.py", line 132, in main
    image_paths = list_images(args.input)
                  ^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/CV2_Final_Project-1/scripts/infer.py", line 27, in list_images
    raise FileNotFoundError(f"Input path does not exist: {path}")
FileNotFoundError: Input path does not exist: data/raw/maps/test/1.jpg
